# KMMMU Evaluation code

In [12]:
!pip -q install -U "vllm" "httpx" "datasets" "pandas" "tqdm" "pillow" "requests" "openai"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.80.7 requires grpcio<1.68.0,>=1.62.3, but you have grpcio 1.78.1 which is incompatible.


## Load KMMMU Dataset

In [1]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("HAERAE-HUB/KMMMU", data_files="kmmmu.csv")
df = ds["train"].to_pandas()

df.head()

,question,answer,question_type,image_link
0,다음 <그림>은 지난 3년 동안 A ~ Q 기업 간에 발생한 소송 관계를 나타낸 것...,4,객관식(단일정답),['https://huggingface.co/datasets/HAERAE-HUB/K...
1,다음 <표>는 A시와 B시의 민원접수 및 처리 현황에 대한 자료이다. 이에 대한 설...,5,객관식(단일정답),['https://huggingface.co/datasets/HAERAE-HUB/K...
2,다음 <표>는 1885 ~ 1892년 동안 조선의 대청ㆍ대일 무역규모를 나타낸 자료...,5,객관식(단일정답),['https://huggingface.co/datasets/HAERAE-HUB/K...
3,다음 <표>는 2011년 국내 6개 유망 벤처기업의 매출액과 CEO 연봉에 대한 자...,"(A): KOREDU, (B): OH케미컬, (C): 과천파밍, (D): GF환경,...",주관식(단답형 텍스트),['https://huggingface.co/datasets/HAERAE-HUB/K...
4,다음 <표>는 ‘갑’팀 구성원 (가 ~ 라)의 보유 역량 및 수행할 작업(A ~ G...,2,객관식(단일정답),['https://huggingface.co/datasets/HAERAE-HUB/K...


## Settings

In [ ]:
# vllm model (이 코드에서는 Qwen2.5-VL-3B-Instruct) 이미 로드되어있다고 가정.
# vllm serve "Qwen/Qwen2.5-VL-3B-Instruct"

In [2]:
import re, base64, asyncio, httpx, requests
from io import BytesIO
from PIL import Image
from tqdm import tqdm
from urllib.parse import urlparse

VLLM_BASE_URL = f"http://localhost:8000/v1"
URL_RE = re.compile(r"https?://[^\s\"'<>]+")
SESSION = requests.Session()

# ===============================
# Prompt
# ===============================
PROMPT_TEMPLATE = """
You are a careful and precise problem-solving assistant.

Instructions:
1. Read the question carefully.
2. If an image is provided, carefully analyze both the question text and the image.
3. Think step by step before reaching the final answer.
4. Provide a clear and concise explanation in fluent Korean.

Final Answer Format (STRICT):
- The final answer must appear exactly once.
- It must be enclosed inside LaTeX \\boxed{...}.
- Do NOT use alternative formats such as "Answer:", "Final Answer:", or any other concluding phrases.
- Only the content inside \\boxed{...} will be evaluated.

Output Structure:
- Explanation in Korean (if needed)
- Final answer in the format: \\boxed{your_answer}

Problem:
"""

# =============================

def parse_url_list(raw):
    if pd.isna(raw):
        return []
    urls = URL_RE.findall(str(raw))
    out, seen = [], set()
    for u in urls:
        p = urlparse(u)
        if p.scheme and p.netloc and u not in seen:
            out.append(u); seen.add(u)
    return out

def url_to_data_uri(url: str, max_side: int = 768):
    r = SESSION.get(url, timeout=15)
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert("RGB")
    # img.thumbnail((max_side, max_side))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=100)
    b64 = base64.b64encode(buf.getvalue()).decode()
    return f"data:image/jpeg;base64,{b64}"

def save_now():
    tmp = OUT_PATH + ".tmp"
    df.drop(columns=["_image_list"], errors="ignore").to_csv(tmp, index=False)
    os.replace(tmp, OUT_PATH)

async def build_mm_messages(question: str, image_urls: list[str]):
    content = [{"type": "text", "text": PROMPT_TEMPLATE + question}]
    for url in (image_urls or []):
        try:
            data_uri = await asyncio.to_thread(url_to_data_uri, url)
            content.append({"type": "image_url", "image_url": {"url": data_uri}})
        except Exception:
            continue
    return [{"role": "user", "content": content}]

async def vllm_chat(http: httpx.AsyncClient, messages, model: str, max_tokens=1024):
    url = f"{VLLM_BASE_URL}/chat/completions"
    payload = {
        "model": model,
        "messages": messages,
        "temperature": 1.0,
        "top_p": 1.0,
        "max_tokens": max_tokens,
    }
    r = await http.post(url, json=payload)
    r.raise_for_status()
    data = r.json()
    return (data["choices"][0]["message"].get("content") or "").strip()

## Generating Response

In [23]:
import os, httpx
from tqdm import tqdm

OUT_PATH = "KMMMU_VLLM_OUTPUT.csv"
MAX_TOKENS = 4096
SAVE_EVERY = 16
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"

df["_image_list"] = df["image_link"].apply(parse_url_list) if "image_link" in df.columns else [[]]*len(df)
if "response" not in df.columns:
    df["response"] = ""

async def run_generation(n=0):
    n = len(df) if n == 0 else min(n, len(df))
    idxs = [i for i in range(n) if not str(df.at[i, "response"]).strip()]

    timeout = httpx.Timeout(60.0, read=300.0)
    async with httpx.AsyncClient(timeout=timeout) as http:
        for k, i in enumerate(tqdm(idxs), 1):
            try:
                msgs = await build_mm_messages(df.at[i, "question"], df.at[i, "_image_list"])
                txt = await vllm_chat(http, msgs, model=MODEL, max_tokens=MAX_TOKENS)
            except Exception as e:
                txt = f"[ERROR] {type(e).__name__}: {e}"

            df.at[i, "response"] = txt

            if SAVE_EVERY and k % SAVE_EVERY == 0:
                save_now()

    save_now()

# 테스트 32개
await run_generation(n=32)

# await run_generation(n=0)
print("saved:", OUT_PATH)

0it [00:00, ?it/s]

saved: KMMMU_VLLM_OUTPUT.csv


In [3]:
# asyncio code

import os, asyncio, httpx
from tqdm import tqdm

OUT_PATH = "KMMMU_VLLM_OUTPUT.csv"
MAX_TOKENS = 4096
CONCURRENCY = 16
SAVE_EVERY = 16
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"


# image list + response column
df["_image_list"] = df["image_link"].apply(parse_url_list) if "image_link" in df.columns else [[]]*len(df)

if "response" not in df.columns:
    df["response"] = ""

async def run_generation(n=0):
    n = len(df) if n == 0 else min(n, len(df))
    idxs = [i for i in range(n) if not str(df.at[i, "response"]).strip()]

    pbar = tqdm(total=len(idxs))
    save_lock = asyncio.Lock()
    done = 0

    async def do_one(http, i):
        nonlocal done
        try:
            msgs = await build_mm_messages(df.at[i, "question"], df.at[i, "_image_list"])
            txt = await vllm_chat(http, msgs, model=MODEL, max_tokens=MAX_TOKENS)
        except Exception as e:
            txt = f"[ERROR] {type(e).__name__}: {e}"
        df.at[i, "response"] = txt

        done += 1
        pbar.update(1)

        if SAVE_EVERY and done % SAVE_EVERY == 0:
            async with save_lock:
                save_now()

    timeout = httpx.Timeout(60.0, read=300.0)
    limits = httpx.Limits(max_connections=CONCURRENCY*2, max_keepalive_connections=CONCURRENCY*2)

    async with httpx.AsyncClient(timeout=timeout, limits=limits) as http:
        sem = asyncio.Semaphore(CONCURRENCY)

        async def bound(i):
            async with sem:
                await do_one(http, i)

        await asyncio.gather(*[bound(i) for i in idxs])

    pbar.close()
    save_now()

# test run (n 개수 지정)
await run_generation(n=32)

# 전체: n=0
# await run_generation(n=0)

print("saved:", OUT_PATH)

100%|█████████████████████████████████████████████████████████████████████| 32/32 [00:22<00:00,  1.40it/s]

saved: KMMMU_VLLM_OUTPUT.csv


## Parsing Answer

In [20]:
import re

BOX_RE = re.compile(r"\\boxed\{(.+?)\}", re.DOTALL)

CIRCLED_NUM = {
    "①":"1","②":"2","③":"3","④":"4","⑤":"5",
    "⑥":"6","⑦":"7","⑧":"8","⑨":"9","⑩":"10",
}

def parse_answer(resp: str) -> str:
    """
    Return the final answer string inside the last \\boxed{...}.
    - Converts circled numbers (①②③...) to plain digits.
    """
    if not isinstance(resp, str):
        return ""

    m = BOX_RE.findall(resp)
    if not m:
        return ""

    ans = m[-1].strip()

    if ans in CIRCLED_NUM:
        ans = CIRCLED_NUM[ans]

    return ans
    
df["parsed_response"] = df["response"].apply(parse_answer)
df[["response","parsed_response"]].head()

,response,parsed_response
0,\boxed{③},3
1,② '(' mest ' 수용'in 트arsi出品聯発案유資' litre depetта...,
2,해당 문제의 마지막 표현은 다음과 같습니다. 이를 바탕으로 풀어하겠습니다.\n\n⑤...,
3,\boxed{(B)},(B)
4,\boxed{⑤},5


## Judge with LLM

In [21]:
import os
import json
from openai import OpenAI

# os.environ["OPENAI_API_KEY"] = "Your_api_key"

client = OpenAI()

JUDGE_MODEL = "gpt-5-mini"

def judge_one(gold: str, pred: str):
    # pred가 비어있으면 바로 오답 처리
    if not str(pred).strip():
        return json.dumps({"label": False})

    prompt = f"""
        You are a strict answer equivalence judge.

    Your task:
    Determine whether the predicted answer (PRED) is mathematically or logically equivalent to the gold answer (GOLD).
    
    Rules:
    - Only compare the final answers.
    - Ignore formatting differences (e.g., 0.5 vs 1/2, whitespace, parentheses).
    - Do not consider explanation text.
    - If they are equivalent, return label "True".
    - Otherwise return label "False".

    Return ONLY a valid JSON object with exactly: label (True/False)
    Do not include markdown, code blocks, or extra text.
    
    GOLD: {gold}
    PRED: {pred}
    """

    r = client.responses.create(
        model=JUDGE_MODEL,
        input=prompt,
        text={"format": {"type": "json_object"}},
    )
    return r.output_text

if "is_correct" not in df.columns:
    df["is_correct"] = ""

def safe_parse_json(s):
    try:
        return json.loads(s)
    except Exception:
        return {"label": ""}

In [22]:
# Judge
# for i in tqdm(range(len(df))):
for i in tqdm(range(5)):
    out = judge_one(df.at[i, "answer"], df.at[i, "parsed_response"])
    j = safe_parse_json(out)
    df.at[i, "is_correct"] = j.get("label", "")
    print(i, out, "=> saved is_correct:", df.at[i, "is_correct"])

df.to_csv("KMMMU_VLLM_JUDGED.csv", index=False)
print("Saved!")

 20%|██████████████▏                                                        | 1/5 [00:01<00:06,  1.71s/it]

0 {"label":"False"} => saved is_correct: False
1 {"label": false} => saved is_correct: False
2 {"label": false} => saved is_correct: False


 80%|████████████████████████████████████████████████████████▊              | 4/5 [00:05<00:01,  1.47s/it]

3 {"label":"False"} => saved is_correct: False


100%|███████████████████████████████████████████████████████████████████████| 5/5 [00:07<00:00,  1.55s/it]

4 {"label":"False"} => saved is_correct: False
Saved!
